In [ ]:
%%sql -r dataframe_1
-- =====================================================
-- PHASE 5A: OPS TABLES
-- NOTEBOOK: NB_05_DATA_QUALITY_RECONCILIATION
-- =====================================================

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;
USE SCHEMA OPS;

-- 1. Ingestion audit
CREATE TABLE IF NOT EXISTS OPS.INGESTION_AUDIT (
    AUDIT_ID              NUMBER AUTOINCREMENT,
    BATCH_ID              VARCHAR(100),
    SOURCE_TYPE           VARCHAR(50),
    SOURCE_FILE_NAME      VARCHAR(500),
    FILE_TIMESTAMP        TIMESTAMP_NTZ,
    INGESTION_TIMESTAMP   TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    RECORD_COUNT          NUMBER(38,0),
    SUCCESS_COUNT         NUMBER(38,0),
    REJECTED_COUNT        NUMBER(38,0),
    STATUS                VARCHAR(30),
    ERROR_MESSAGE         VARCHAR(2000)
);

-- 2. Batch control
CREATE TABLE IF NOT EXISTS OPS.BATCH_CONTROL (
    BATCH_ID              VARCHAR(100),
    SOURCE_TYPE           VARCHAR(50),
    SOURCE_FILE_NAME      VARCHAR(500),
    FILE_TIMESTAMP        TIMESTAMP_NTZ,
    BATCH_START_TIMESTAMP TIMESTAMP_NTZ,
    BATCH_END_TIMESTAMP   TIMESTAMP_NTZ,
    STATUS                VARCHAR(30),
    RECORD_COUNT          NUMBER(38,0),
    PROCESSED_COUNT       NUMBER(38,0),
    REJECTED_COUNT        NUMBER(38,0)
);

-- 3. Rejected member records
CREATE TABLE IF NOT EXISTS OPS.REJECTED_MEMBER_RECORDS (
    REJECT_ID             NUMBER AUTOINCREMENT,
    BATCH_ID              VARCHAR(100),
    SOURCE_FILE_NAME      VARCHAR(500),
    SOURCE_ROW_NUMBER     NUMBER(38,0),
    RAW_RECORD            VARIANT,
    REJECTION_REASON      VARCHAR(2000),
    REJECTION_TIMESTAMP   TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- 4. Rejected redemption records
CREATE TABLE IF NOT EXISTS OPS.REJECTED_REDEMPTION_RECORDS (
    REJECT_ID             NUMBER AUTOINCREMENT,
    BATCH_ID              VARCHAR(100),
    SOURCE_FILE_NAME      VARCHAR(500),
    SOURCE_RECORD_NUMBER  NUMBER(38,0),
    RAW_RECORD            VARIANT,
    REJECTION_REASON      VARCHAR(2000),
    REJECTION_TIMESTAMP   TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- 5. Data quality results
CREATE TABLE IF NOT EXISTS OPS.DATA_QUALITY_RESULTS (
    RESULT_ID             NUMBER AUTOINCREMENT,
    BATCH_ID              VARCHAR(100),
    SOURCE_TYPE           VARCHAR(50),
    TABLE_NAME            VARCHAR(255),
    RULE_NAME             VARCHAR(255),
    TOTAL_RECORDS         NUMBER(38,0),
    PASSED_RECORDS        NUMBER(38,0),
    FAILED_RECORDS        NUMBER(38,0),
    CHECK_STATUS          VARCHAR(30),
    EXECUTION_TIMESTAMP   TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Verify
SHOW TABLES IN SCHEMA OPS;

In [ ]:
%%sql -r dataframe_2
-- =====================================================
-- PHASE 5B: MART DIMENSIONAL TABLES
-- =====================================================

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;
USE SCHEMA MART;

-- 1. Date dimension
CREATE TABLE IF NOT EXISTS MART.DIM_DATE (
    DATE_KEY          NUMBER(8,0),
    FULL_DATE         DATE,
    DAY_NUMBER        NUMBER(2,0),
    DAY_NAME          VARCHAR(20),
    WEEK_NUMBER       NUMBER(2,0),
    MONTH_NUMBER      NUMBER(2,0),
    MONTH_NAME        VARCHAR(20),
    QUARTER_NUMBER    NUMBER(1,0),
    YEAR_NUMBER      NUMBER(4,0),
    IS_WEEKEND        BOOLEAN
);

-- 2. Member dimension
CREATE TABLE IF NOT EXISTS MART.DIM_MEMBER (
    MEMBER_KEY          NUMBER AUTOINCREMENT,
    MEMBER_ID           VARCHAR(18),
    MEMBER_NAME         VARCHAR(255),
    ENROLLMENT_DATE     DATE,
    LAST_FLIGHT_DATE    DATE,
    TIER_CODE           VARCHAR(5),
    STATE               VARCHAR(5),
    COUNTRY             VARCHAR(5),
    AGE                 NUMBER(3,0),
    ACTIVE_MEMBER       CHAR(1),
    STALE_MEMBER        BOOLEAN,
    SOURCE_FILE_NAME    VARCHAR(500),
    FILE_TIMESTAMP      TIMESTAMP_NTZ
);

-- 3. Redemption fact
CREATE TABLE IF NOT EXISTS MART.FACT_REDEMPTION (
    REDEMPTION_KEY       NUMBER AUTOINCREMENT,
    DATE_KEY             NUMBER(8,0),
    MEMBER_KEY           NUMBER,
    MEMBER_ID            VARCHAR(18),
    TXN_ID               VARCHAR(100),
    TXN_DATE             DATE,
    PARTNER              VARCHAR(255),
    MILES_REDEEMED       NUMBER(18,0),
    STATUS               VARCHAR(50),
    SOURCE_FILE_NAME     VARCHAR(500),
    BATCH_ID             VARCHAR(100)
);

-- Verify
SHOW TABLES IN SCHEMA MART;

DESCRIBE TABLE MART.DIM_DATE;
DESCRIBE TABLE MART.DIM_MEMBER;
DESCRIBE TABLE MART.FACT_REDEMPTION;

In [ ]:

-- =====================================================
-- CELL 3: END-TO-END DATA RECONCILIATION
-- =====================================================

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;

-- 1. MEMBER RECORD RECONCILIATION
SELECT
    'MEMBER' AS DATASET,
    'RAW' AS LAYER,
    COUNT(*) AS RECORD_COUNT
FROM RAW.RAW_MEMBER_PROFILES

UNION ALL

SELECT
    'MEMBER',
    'STG',
    COUNT(*)
FROM STG.STG_MEMBER_PROFILES

UNION ALL

SELECT
    'MEMBER',
    'CORE',
    COUNT(*)
FROM CORE.MEMBER_PROFILE;


-- 2. REDEMPTION RECORD RECONCILIATION
-- RAW counts JSON documents; STG and CORE count flattened transactions.
SELECT
    'REDEMPTION' AS DATASET,
    'RAW_JSON_DOCUMENTS' AS LAYER,
    COUNT(*) AS RECORD_COUNT
FROM RAW.RAW_REDEMPTIONS

UNION ALL

SELECT
    'REDEMPTION',
    'STG_TRANSACTIONS',
    COUNT(*)
FROM STG.STG_REDEMPTION_TRANSACTIONS

UNION ALL

SELECT
    'REDEMPTION',
    'CORE_TRANSACTIONS',
    COUNT(*)
FROM CORE.REDEMPTION_TRANSACTIONS;


-- 3. MEMBER VALIDATION SUMMARY
SELECT
    VALIDATION_STATUS,
    COUNT(*) AS RECORD_COUNT
FROM STG.STG_MEMBER_PROFILES
GROUP BY VALIDATION_STATUS
ORDER BY VALIDATION_STATUS;


-- 4. REDEMPTION VALIDATION AND BUSINESS SUMMARY
SELECT
    COUNT(*) AS TOTAL_TRANSACTIONS,
    COUNT_IF(VALIDATION_STATUS = 'VALID') AS VALID_TRANSACTIONS,
    COUNT_IF(VALIDATION_STATUS = 'REJECTED') AS REJECTED_TRANSACTIONS,
    SUM(MILES_REDEEMED) AS TOTAL_MILES_REDEEMED,
    COUNT(DISTINCT MEMBER_ID) AS DISTINCT_MEMBERS
FROM STG.STG_REDEMPTION_TRANSACTIONS;


-- 5. COUNTRY-WISE MEMBER DISTRIBUTION
SELECT
    COUNTRY,
    COUNT(*) AS MEMBER_COUNT
FROM CORE.MEMBER_PROFILE
GROUP BY COUNTRY
ORDER BY COUNTRY;


-- 6. REDEMPTION TO MEMBER MATCHING
SELECT
    COUNT(*) AS TOTAL_TRANSACTIONS,
    COUNT_IF(M.MEMBER_ID IS NOT NULL) AS MATCHED_TRANSACTIONS,
    COUNT_IF(M.MEMBER_ID IS NULL) AS UNMATCHED_TRANSACTIONS
FROM CORE.REDEMPTION_TRANSACTIONS R
LEFT JOIN CORE.MEMBER_PROFILE M
    ON R.MEMBER_ID = M.MEMBER_ID;


In [ ]:
%%sql -r dataframe_4
DESC TABLE OPS.DATA_QUALITY_RESULTS;

In [ ]:

-- =====================================================
-- CELL 4: STORE DATA QUALITY RESULTS
-- DYNAMIC BATCH-BASED QUALITY LOGGING
-- =====================================================

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;

BEGIN TRANSACTION;

-- Remove previous quality results for batches
-- that are being processed again.
DELETE FROM OPS.DATA_QUALITY_RESULTS
WHERE BATCH_ID IN (
    SELECT BATCH_ID
    FROM STG.STG_MEMBER_PROFILES

    UNION

    SELECT BATCH_ID
    FROM STG.STG_REDEMPTION_TRANSACTIONS
);

INSERT INTO OPS.DATA_QUALITY_RESULTS (
    BATCH_ID,
    SOURCE_TYPE,
    TABLE_NAME,
    RULE_NAME,
    TOTAL_RECORDS,
    PASSED_RECORDS,
    FAILED_RECORDS,
    CHECK_STATUS
)

WITH
MEMBER_BATCHES AS (
    SELECT DISTINCT BATCH_ID
    FROM STG.STG_MEMBER_PROFILES
),

REDEMPTION_BATCHES AS (
    SELECT DISTINCT BATCH_ID
    FROM STG.STG_REDEMPTION_TRANSACTIONS
),

CHECKS AS (

    -- 1. Member mandatory-field validation
    SELECT
        B.BATCH_ID,
        'MEMBER' AS SOURCE_TYPE,
        'STG.STG_MEMBER_PROFILES' AS TABLE_NAME,
        'MANDATORY_FIELDS' AS RULE_NAME,
        COUNT(*) AS TOTAL_RECORDS,
        COUNT_IF(S.VALIDATION_STATUS = 'VALID') AS PASSED_RECORDS,
        COUNT_IF(S.VALIDATION_STATUS = 'REJECTED') AS FAILED_RECORDS
    FROM MEMBER_BATCHES B
    JOIN STG.STG_MEMBER_PROFILES S
        ON B.BATCH_ID = S.BATCH_ID
    GROUP BY B.BATCH_ID

    UNION ALL

    -- 2. Member ID uniqueness
    SELECT
        B.BATCH_ID,
        'MEMBER',
        'STG.STG_MEMBER_PROFILES',
        'MEMBER_ID_UNIQUENESS',
        COUNT(*),
        COUNT(DISTINCT S.MEMBER_ID),
        COUNT(*) - COUNT(DISTINCT S.MEMBER_ID)
    FROM MEMBER_BATCHES B
    JOIN STG.STG_MEMBER_PROFILES S
        ON B.BATCH_ID = S.BATCH_ID
    WHERE S.VALIDATION_STATUS = 'VALID'
    GROUP BY B.BATCH_ID

    UNION ALL

    -- 3. Member name uniqueness
    SELECT
        B.BATCH_ID,
        'MEMBER',
        'STG.STG_MEMBER_PROFILES',
        'MEMBER_NAME_UNIQUENESS',
        COUNT(*),
        COUNT(DISTINCT S.MEMBER_NAME),
        COUNT(*) - COUNT(DISTINCT S.MEMBER_NAME)
    FROM MEMBER_BATCHES B
    JOIN STG.STG_MEMBER_PROFILES S
        ON B.BATCH_ID = S.BATCH_ID
    WHERE S.VALIDATION_STATUS = 'VALID'
    GROUP BY B.BATCH_ID

    UNION ALL

    -- 4. Supported country-code validation
    SELECT
        B.BATCH_ID,
        'MEMBER',
        'STG.STG_MEMBER_PROFILES',
        'COUNTRY_CODE_VALIDATION',
        COUNT(*),
        COUNT_IF(S.COUNTRY IN ('IND', 'USA', 'PHIL', 'CAN', 'AU')),
        COUNT_IF(
            S.COUNTRY IS NULL
            OR S.COUNTRY NOT IN ('IND', 'USA', 'PHIL', 'CAN', 'AU')
        )
    FROM MEMBER_BATCHES B
    JOIN STG.STG_MEMBER_PROFILES S
        ON B.BATCH_ID = S.BATCH_ID
    WHERE S.VALIDATION_STATUS = 'VALID'
    GROUP BY B.BATCH_ID

    UNION ALL

    -- 5. Redemption mandatory-field validation
    SELECT
        B.BATCH_ID,
        'REDEMPTION',
        'STG.STG_REDEMPTION_TRANSACTIONS',
        'MANDATORY_FIELDS',
        COUNT(*),
        COUNT_IF(S.VALIDATION_STATUS = 'VALID'),
        COUNT_IF(S.VALIDATION_STATUS = 'REJECTED')
    FROM REDEMPTION_BATCHES B
    JOIN STG.STG_REDEMPTION_TRANSACTIONS S
        ON B.BATCH_ID = S.BATCH_ID
    GROUP BY B.BATCH_ID

    UNION ALL

    -- 6. Redemption transaction-key uniqueness
    SELECT
        B.BATCH_ID,
        'REDEMPTION',
        'STG.STG_REDEMPTION_TRANSACTIONS',
        'TRANSACTION_KEY_UNIQUENESS',
        COUNT(*),
        COUNT(DISTINCT S.MEMBER_ID || '|' || S.TXN_ID),
        COUNT(*) - COUNT(DISTINCT S.MEMBER_ID || '|' || S.TXN_ID)
    FROM REDEMPTION_BATCHES B
    JOIN STG.STG_REDEMPTION_TRANSACTIONS S
        ON B.BATCH_ID = S.BATCH_ID
    WHERE S.VALIDATION_STATUS = 'VALID'
    GROUP BY B.BATCH_ID

    UNION ALL

    -- 7. Redemption member-reference validation
    SELECT
        B.BATCH_ID,
        'REDEMPTION',
        'STG.STG_REDEMPTION_TRANSACTIONS',
        'MEMBER_REFERENCE_VALIDATION',
        COUNT(*),
        COUNT_IF(M.MEMBER_ID IS NOT NULL),
        COUNT_IF(M.MEMBER_ID IS NULL)
    FROM REDEMPTION_BATCHES B
    JOIN STG.STG_REDEMPTION_TRANSACTIONS R
        ON B.BATCH_ID = R.BATCH_ID
    LEFT JOIN CORE.MEMBER_PROFILE M
        ON R.MEMBER_ID = M.MEMBER_ID
    WHERE R.VALIDATION_STATUS = 'VALID'
    GROUP BY B.BATCH_ID
)

SELECT
    BATCH_ID,
    SOURCE_TYPE,
    TABLE_NAME,
    RULE_NAME,
    TOTAL_RECORDS,
    PASSED_RECORDS,
    FAILED_RECORDS,
    CASE
        WHEN TOTAL_RECORDS = 0 THEN 'NO_DATA'
        WHEN FAILED_RECORDS = 0 THEN 'PASSED'
        ELSE 'FAILED'
    END AS CHECK_STATUS
FROM CHECKS;

COMMIT;

-- Verify persisted results for all batches
SELECT *
FROM OPS.DATA_QUALITY_RESULTS
ORDER BY BATCH_ID, SOURCE_TYPE, RULE_NAME;


In [ ]:

-- =====================================================
-- CELL 5: OPERATIONAL AUDIT AND BATCH CONTROL
-- DYNAMIC BATCH-BASED OPERATIONAL LOGGING
-- =====================================================

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;

BEGIN TRANSACTION;


-- 1. INGESTION AUDIT
-- Discover all batches from RAW and obtain processing
-- counts from STG.

MERGE INTO OPS.INGESTION_AUDIT T
USING (
    WITH RAW_BATCHES AS (

        SELECT
            BATCH_ID,
            'MEMBER' AS SOURCE_TYPE,
            SOURCE_FILE_NAME,
            MAX(FILE_TIMESTAMP) AS FILE_TIMESTAMP
        FROM RAW.RAW_MEMBER_PROFILES
        GROUP BY BATCH_ID, SOURCE_FILE_NAME

        UNION ALL

        SELECT
            BATCH_ID,
            'REDEMPTION',
            SOURCE_FILE_NAME,
            MAX(FILE_TIMESTAMP)
        FROM RAW.RAW_REDEMPTIONS
        GROUP BY BATCH_ID, SOURCE_FILE_NAME
    ),

    MEMBER_COUNTS AS (
        SELECT
            BATCH_ID,
            COUNT(*) AS RECORD_COUNT,
            COUNT_IF(VALIDATION_STATUS = 'VALID') AS SUCCESS_COUNT,
            COUNT_IF(VALIDATION_STATUS = 'REJECTED') AS REJECTED_COUNT
        FROM STG.STG_MEMBER_PROFILES
        GROUP BY BATCH_ID
    ),

    REDEMPTION_COUNTS AS (
        SELECT
            BATCH_ID,
            COUNT(*) AS RECORD_COUNT,
            COUNT_IF(VALIDATION_STATUS = 'VALID') AS SUCCESS_COUNT,
            COUNT_IF(VALIDATION_STATUS = 'REJECTED') AS REJECTED_COUNT
        FROM STG.STG_REDEMPTION_TRANSACTIONS
        GROUP BY BATCH_ID
    ),

    PROCESSING_COUNTS AS (
        SELECT
            BATCH_ID,
            'MEMBER' AS SOURCE_TYPE,
            RECORD_COUNT,
            SUCCESS_COUNT,
            REJECTED_COUNT
        FROM MEMBER_COUNTS

        UNION ALL

        SELECT
            BATCH_ID,
            'REDEMPTION',
            RECORD_COUNT,
            SUCCESS_COUNT,
            REJECTED_COUNT
        FROM REDEMPTION_COUNTS
    )

    SELECT
        R.BATCH_ID,
        R.SOURCE_TYPE,
        R.SOURCE_FILE_NAME,
        R.FILE_TIMESTAMP,
        COALESCE(P.RECORD_COUNT, 0) AS RECORD_COUNT,
        COALESCE(P.SUCCESS_COUNT, 0) AS SUCCESS_COUNT,
        COALESCE(P.REJECTED_COUNT, 0) AS REJECTED_COUNT,
        CASE
            WHEN P.BATCH_ID IS NULL THEN 'PENDING'
            WHEN P.REJECTED_COUNT > 0 THEN 'PARTIAL_SUCCESS'
            ELSE 'SUCCESS'
        END AS STATUS
    FROM RAW_BATCHES R
    LEFT JOIN PROCESSING_COUNTS P
        ON R.BATCH_ID = P.BATCH_ID
       AND R.SOURCE_TYPE = P.SOURCE_TYPE

) S
ON T.BATCH_ID = S.BATCH_ID
AND T.SOURCE_TYPE = S.SOURCE_TYPE
AND T.SOURCE_FILE_NAME = S.SOURCE_FILE_NAME

WHEN MATCHED THEN UPDATE SET
    T.FILE_TIMESTAMP = S.FILE_TIMESTAMP,
    T.RECORD_COUNT = S.RECORD_COUNT,
    T.SUCCESS_COUNT = S.SUCCESS_COUNT,
    T.REJECTED_COUNT = S.REJECTED_COUNT,
    T.STATUS = S.STATUS,
    T.ERROR_MESSAGE = NULL

WHEN NOT MATCHED THEN INSERT (
    BATCH_ID,
    SOURCE_TYPE,
    SOURCE_FILE_NAME,
    FILE_TIMESTAMP,
    RECORD_COUNT,
    SUCCESS_COUNT,
    REJECTED_COUNT,
    STATUS
)
VALUES (
    S.BATCH_ID,
    S.SOURCE_TYPE,
    S.SOURCE_FILE_NAME,
    S.FILE_TIMESTAMP,
    S.RECORD_COUNT,
    S.SUCCESS_COUNT,
    S.REJECTED_COUNT,
    S.STATUS
);


-- 2. BATCH CONTROL
-- One control record per source batch.

MERGE INTO OPS.BATCH_CONTROL T
USING (
    WITH MEMBER_BATCHES AS (
        SELECT
            BATCH_ID,
            SOURCE_FILE_NAME,
            MAX(FILE_TIMESTAMP) AS FILE_TIMESTAMP,
            MIN(INGESTION_TIMESTAMP) AS BATCH_START_TIMESTAMP,
            COUNT(*) AS RECORD_COUNT,
            COUNT_IF(VALIDATION_STATUS = 'VALID') AS PROCESSED_COUNT,
            COUNT_IF(VALIDATION_STATUS = 'REJECTED') AS REJECTED_COUNT
        FROM STG.STG_MEMBER_PROFILES
        GROUP BY BATCH_ID, SOURCE_FILE_NAME
    ),

    REDEMPTION_BATCHES AS (
        SELECT
            BATCH_ID,
            SOURCE_FILE_NAME,
            MAX(FILE_TIMESTAMP) AS FILE_TIMESTAMP,
            MIN(INGESTION_TIMESTAMP) AS BATCH_START_TIMESTAMP,
            COUNT(*) AS RECORD_COUNT,
            COUNT_IF(VALIDATION_STATUS = 'VALID') AS PROCESSED_COUNT,
            COUNT_IF(VALIDATION_STATUS = 'REJECTED') AS REJECTED_COUNT
        FROM STG.STG_REDEMPTION_TRANSACTIONS
        GROUP BY BATCH_ID, SOURCE_FILE_NAME
    )

    SELECT
        BATCH_ID,
        'MEMBER' AS SOURCE_TYPE,
        SOURCE_FILE_NAME,
        FILE_TIMESTAMP,
        BATCH_START_TIMESTAMP,
        RECORD_COUNT,
        PROCESSED_COUNT,
        REJECTED_COUNT
    FROM MEMBER_BATCHES

    UNION ALL

    SELECT
        BATCH_ID,
        'REDEMPTION',
        SOURCE_FILE_NAME,
        FILE_TIMESTAMP,
        BATCH_START_TIMESTAMP,
        RECORD_COUNT,
        PROCESSED_COUNT,
        REJECTED_COUNT
    FROM REDEMPTION_BATCHES

) S
ON T.BATCH_ID = S.BATCH_ID

WHEN MATCHED THEN UPDATE SET
    T.SOURCE_TYPE = S.SOURCE_TYPE,
    T.SOURCE_FILE_NAME = S.SOURCE_FILE_NAME,
    T.FILE_TIMESTAMP = S.FILE_TIMESTAMP,
    T.BATCH_END_TIMESTAMP = CURRENT_TIMESTAMP(),
    T.STATUS = CASE
        WHEN S.REJECTED_COUNT > 0 THEN 'PARTIAL_SUCCESS'
        ELSE 'SUCCESS'
    END,
    T.RECORD_COUNT = S.RECORD_COUNT,
    T.PROCESSED_COUNT = S.PROCESSED_COUNT,
    T.REJECTED_COUNT = S.REJECTED_COUNT

WHEN NOT MATCHED THEN INSERT (
    BATCH_ID,
    SOURCE_TYPE,
    SOURCE_FILE_NAME,
    FILE_TIMESTAMP,
    BATCH_START_TIMESTAMP,
    BATCH_END_TIMESTAMP,
    STATUS,
    RECORD_COUNT,
    PROCESSED_COUNT,
    REJECTED_COUNT
)
VALUES (
    S.BATCH_ID,
    S.SOURCE_TYPE,
    S.SOURCE_FILE_NAME,
    S.FILE_TIMESTAMP,
    S.BATCH_START_TIMESTAMP,
    CURRENT_TIMESTAMP(),
    CASE
        WHEN S.REJECTED_COUNT > 0 THEN 'PARTIAL_SUCCESS'
        ELSE 'SUCCESS'
    END,
    S.RECORD_COUNT,
    S.PROCESSED_COUNT,
    S.REJECTED_COUNT
);


-- 3. LOG REJECTED MEMBER RECORDS

INSERT INTO OPS.REJECTED_MEMBER_RECORDS (
    BATCH_ID,
    SOURCE_FILE_NAME,
    SOURCE_ROW_NUMBER,
    RAW_RECORD,
    REJECTION_REASON
)
SELECT
    S.BATCH_ID,
    S.SOURCE_FILE_NAME,
    S.SOURCE_ROW_NUMBER,
    R.RAW_RECORD,
    S.REJECTION_REASON
FROM STG.STG_MEMBER_PROFILES S
JOIN RAW.RAW_MEMBER_PROFILES R
    ON S.SOURCE_FILE_NAME = R.SOURCE_FILE_NAME
   AND S.SOURCE_ROW_NUMBER = R.SOURCE_ROW_NUMBER
WHERE S.VALIDATION_STATUS = 'REJECTED'
  AND NOT EXISTS (
      SELECT 1
      FROM OPS.REJECTED_MEMBER_RECORDS X
      WHERE X.BATCH_ID = S.BATCH_ID
        AND X.SOURCE_ROW_NUMBER = S.SOURCE_ROW_NUMBER
  );


-- 4. LOG REJECTED REDEMPTION RECORDS

INSERT INTO OPS.REJECTED_REDEMPTION_RECORDS (
    BATCH_ID,
    SOURCE_FILE_NAME,
    SOURCE_RECORD_NUMBER,
    RAW_RECORD,
    REJECTION_REASON
)
SELECT
    S.BATCH_ID,
    S.SOURCE_FILE_NAME,
    S.SOURCE_RECORD_NUMBER,
    R.RAW_RECORD,
    S.REJECTION_REASON
FROM STG.STG_REDEMPTION_TRANSACTIONS S
JOIN RAW.RAW_REDEMPTIONS R
    ON S.SOURCE_FILE_NAME = R.SOURCE_FILE_NAME
   AND S.SOURCE_RECORD_NUMBER = R.SOURCE_RECORD_NUMBER
WHERE S.VALIDATION_STATUS = 'REJECTED'
  AND NOT EXISTS (
      SELECT 1
      FROM OPS.REJECTED_REDEMPTION_RECORDS X
      WHERE X.BATCH_ID = S.BATCH_ID
        AND X.SOURCE_RECORD_NUMBER = S.SOURCE_RECORD_NUMBER
        AND X.REJECTION_REASON = S.REJECTION_REASON
  );

COMMIT;


-- 5. VERIFY INGESTION AUDIT

SELECT *
FROM OPS.INGESTION_AUDIT
ORDER BY INGESTION_TIMESTAMP DESC;


-- 6. VERIFY BATCH CONTROL

SELECT *
FROM OPS.BATCH_CONTROL
ORDER BY BATCH_START_TIMESTAMP DESC;


-- 7. VERIFY REJECTION COUNTS

SELECT
    'MEMBER' AS SOURCE_TYPE,
    COUNT(*) AS REJECTED_RECORDS
FROM OPS.REJECTED_MEMBER_RECORDS

UNION ALL

SELECT
    'REDEMPTION',
    COUNT(*)
FROM OPS.REJECTED_REDEMPTION_RECORDS;
